In [1]:
# ─────────────────────────────────────────────
# 환경 준비 — 라이브러리 불러오기 + 한글 폰트 + 시드 고정
# ─────────────────────────────────────────────
# 필요 시 아래 주석을 해제해 설치하세요.
!pip install numpy pandas matplotlib seaborn pyarrow -q

import os
import platform
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")   # 학습 중 경고 메시지를 잠시 숨깁니다.

# 재현성: 같은 난수를 항상 같게 만들어 결과가 매번 동일하도록 고정합니다.
np.random.seed(42)

# 한글 폰트 설정 (그래프 안 글자가 깨지지 않도록 운영체제별로 분기)
system = platform.system()
if system == "Darwin":          # macOS
    plt.rcParams["font.family"] = "AppleGothic"
elif system == "Windows":       # Windows
    plt.rcParams["font.family"] = "Malgun Gothic"
else:                            # Linux 등
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 5)
sns.set_style("whitegrid")

# 결과 저장용 임시 폴더 (이 노트북 옆에 'd009_outputs/' 가 만들어집니다)
OUT_DIR = Path("d009_outputs")
OUT_DIR.mkdir(exist_ok=True)

print("준비 완료! 라이브러리 버전을 확인합니다.")
print("numpy :", np.__version__)
print("pandas:", pd.__version__)
print("저장 폴더:", OUT_DIR.resolve())

준비 완료! 라이브러리 버전을 확인합니다.
numpy : 2.4.6
pandas: 3.0.3
저장 폴더: C:\DI\D009\d009_outputs


In [2]:
# 종합 프로젝트용 새 데이터 생성 — 베이커리 체인 "빵셀러" 운영 데이터
np.random.seed(13)
n_days = 90
n_stores = 6
n_rows = n_days * n_stores * 4   # 매장 x 일자 x 시간대(4구간)

stores = [f"S{i:02d}" for i in range(1, n_stores + 1)]
items = ["크루아상", "식빵", "케이크", "샌드위치", "쿠키"]

bakery = pd.DataFrame({
    "store_id": np.tile(stores, n_rows // n_stores)[:n_rows],
    "date_str": np.random.choice([
        "2025-04-01", "2025/04/01", "20250401",
        "2025-05-15", "2025/05/15", "20250515",
        "2025-06-20", "2025/06/20", "20250620",
    ], n_rows),
    "item": np.random.choice(items, n_rows),
    "quantity": np.random.choice([1, 1, 2, 2, 3, 5, 50], n_rows),  # 50은 의심값
    "unit_price": np.random.choice([3500, 4500, 5500, 8500, 12000], n_rows),
})
bakery["revenue"] = bakery["quantity"] * bakery["unit_price"]

# 오염 심기
bakery.loc[np.random.choice(n_rows, 60, replace=False), "revenue"] = np.nan
bakery.loc[5, "revenue"] = -45000  # 환불 또는 실수
bakery.loc[bakery.sample(10, random_state=1).index, "store_id"] = " S01 "  # 공백
bakery.loc[bakery.sample(8, random_state=2).index, "item"] = "케익"        # 표기 혼재('케이크' vs '케익')
bakery = pd.concat([bakery, bakery.iloc[[0, 1, 2, 3]]], ignore_index=True)   # 중복 4건

print("빵셀러 데이터 생성 완료:", bakery.shape)
bakery.head()

빵셀러 데이터 생성 완료: (2164, 6)


,store_id,date_str,item,quantity,unit_price,revenue
0,S01,20250401,쿠키,2,12000,24000.0
1,S02,2025-04-01,케이크,2,12000,24000.0
2,S03,2025-04-01,쿠키,1,5500,5500.0
3,S04,2025-06-20,쿠키,2,8500,17000.0
4,S05,20250401,샌드위치,5,4500,22500.0


In [ ]:
# 품질 리포트 함수 v1 — 결측·중복·타입을 한 번에 보여주는 진단기
def quality_report(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''데이터프레임의 품질을 컬럼별로 진단해 한 표로 돌려줍니다.'''
    n_rows = len(df)
    report = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
        "sample": [df[col].dropna().iloc[0] if df[col].notna().any() else None for col in df.columns],
    })
    print(f"[품질 리포트] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  메모리: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    return report

In [10]:
# 품질 리포트 함수 v2 — 수치형 컬럼에 IQR 이상치 비율을 추가
def quality_report_full(df: pd.DataFrame, name: str = "df") -> pd.DataFrame:
    '''v1에 수치형 이상치 비율(IQR)과 의심 타입 컬럼 표시를 추가합니다.'''
    n_rows = len(df)
    base = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2),
        "n_unique": df.nunique(dropna=True),
    })

    # IQR 이상치 비율 (수치형 컬럼만)
    outlier_pct = {}
    for col in df.select_dtypes(include="number").columns:
        s = df[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outlier_pct[col] = ((s < lo) | (s > hi)).mean() * 100
    base["outlier_pct_iqr"] = pd.Series(outlier_pct).round(2)

    # object 컬럼이 실제로는 날짜로 파싱되는지 의심 표시
    suspicious_datetime = []
    for col in df.select_dtypes(include="object").columns:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce")
            if parsed.notna().mean() > 0.8:
                suspicious_datetime.append(col)
        except Exception:
            pass
    base["maybe_datetime"] = base.index.isin(suspicious_datetime)

    print(f"[품질 리포트(완전판)] {name}")
    print(f"  행 수: {n_rows:,}  /  열 수: {len(df.columns)}")
    print(f"  완전 중복 행: {df.duplicated().sum()}건")
    if suspicious_datetime:
        print(f"  📌 날짜로 보이는 object 컬럼: {suspicious_datetime}")
    return base


In [11]:
# 시나리오1 - 품질 질단
qr_bakery = quality_report_full(bakery, "bakery")
qr_bakery

# 결측이 가장 높은 칼럼은 revenue
# maybe_datetime이 True로 잡힌 컬럼은 없으나 컬럼명에서 알 수 있다 싶이 date_str이 datetime일 가능성 있음
# 수치형 중 IQR 이상치 비율이 가장 높은 컬럼은 quantity이다. revenue는 결측이 있어 추후에 다시 확인해야 할 필요 있음

[품질 리포트(완전판)] bakery
  행 수: 2,164  /  열 수: 6
  완전 중복 행: 321건


,dtype,missing,missing_pct,n_unique,outlier_pct_iqr,maybe_datetime
store_id,str,0,0.00,7,NaN,False
date_str,str,0,0.00,9,NaN,False
item,str,0,0.00,6,NaN,False
quantity,int64,0,0.00,5,14.56,False
unit_price,int64,0,0.00,5,0.00,False
revenue,float64,60,2.77,26,17.30,False


In [15]:
# 시나리오 2 — 정제 파이프라인
def b_dedup(df): return df.drop_duplicates().reset_index(drop=True)  # 완전 중복 제거
def b_strip_store(df): return df.assign(store_id=df["store_id"].str.strip())  # store_id의 앞뒤 공백 제거
def b_unify_item(df): return df.assign(item=df["item"].replace({"케익": "케이크"}))  # item의 '케익' → '케이크'로 통일
def b_parse_date(df): return df.assign(  # date_str을 date 컬럼(datetime)으로 변환
    date=pd.to_datetime(df["date_str"], format="mixed", errors="coerce")
).drop(columns=["date_str"])
def b_recalc_revenue(df): return df.assign(  # revenue 결측치를 quantity * unit_price로 재계산
    revenue=df["revenue"].fillna(df["quantity"] * df["unit_price"])
)

refunds_bakery = bakery[bakery["revenue"] < 0].copy()  # revenue < 0인 행은 refunds_bakery로 분리

bakery_clean = (
    bakery
    .pipe(b_dedup)
    .pipe(b_strip_store)
    .pipe(b_unify_item)
    .pipe(b_parse_date)
    .pipe(b_recalc_revenue)                  # revenue 결측 → 재계산으로 복구
    .pipe(lambda d: d[d["revenue"] >= 0])    # 환불 분리 후 본 분석에서 제외
    .reset_index(drop=True)
)

print(f"정제 전: {bakery.shape}  →  정제 후: {bakery_clean.shape}")
print(f"환불 보관: {len(refunds_bakery)}건")
print(f"item 종류: {bakery_clean['item'].unique()}")

revenue 결측 개수: 60
정제 전: (2164, 6)  →  정제 후: (1842, 6)
환불 보관: 1건
item 종류: <ArrowStringArray>
['쿠키', '케이크', '샌드위치', '크루아상', '식빵']
Length: 5, dtype: str


In [16]:
# 시나리오 3 — 변환 + Parquet 저장

# KPI 1: 월별·매장별 매출 합계 (wide)
bakery_clean["month"] = bakery_clean["date"].dt.to_period("M").astype(str)
store_month = (
    bakery_clean.groupby(["store_id", "month"])["revenue"].sum().unstack(fill_value=0).round(0)
)
print("월별·매장별 매출:")
display(store_month)

# KPI 2: 상품별 평균 단가·총매출
item_kpi = (
    bakery_clean.groupby("item")
    .agg(avg_price=("unit_price", "mean"), total_revenue=("revenue", "sum"), n_orders=("revenue", "count"))
    .round(0)
    .sort_values("total_revenue", ascending=False)
)
print("\n상품별 KPI:")
display(item_kpi)

# Parquet 저장
store_month.to_parquet(OUT_DIR / "bakery_store_month.parquet")
item_kpi.to_parquet(OUT_DIR / "bakery_item_kpi.parquet")
print(f"\nParquet 저장 완료: {OUT_DIR.resolve()}")

월별·매장별 매출:


month,2025-04,2025-05,2025-06
store_id,,,
S01,6869500.0,7554000.0,6015500.0
S02,8041500.0,7052500.0,8051000.0
S03,7858500.0,4592500.0,7336000.0
S04,6503000.0,7148000.0,6784000.0
S05,8745000.0,4396500.0,5011500.0
S06,9759500.0,5135000.0,6825000.0



상품별 KPI:


,avg_price,total_revenue,n_orders
item,,,
쿠키,6874.0,26812000.0,373
샌드위치,6560.0,26643500.0,377
케이크,6858.0,25285500.0,373
식빵,6500.0,23814000.0,368
크루아상,6963.0,21123500.0,351



Parquet 저장 완료: C:\DI\D009\d009_outputs
